In [ ]:
%%capture
%pip install ultralytics roboflow

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="API-key")
project = rf.workspace("ufvmestrado").project("abelhas-v3")
version = project.version(1)
dataset = version.download("yolo26")

In [ ]:
dataset_paht = "/content/Abelhas-V3-1"
projeto_path = "/content/drive/MyDrive/Lesc-Bee videos/modelo_rt_dert"
treino_name = "rt_dert_28_02_2026"

In [ ]:
import yaml
import os

# Caminho exato onde o arquivo será salvo no seu Drive
caminho_pasta = "/content/drive/MyDrive/Lesc-Bee videos/modelo_v2"
caminho_arquivo = f"{caminho_pasta}/hyp_rtdetr.yaml"

# Cria a pasta caso ela ainda não exista
os.makedirs(caminho_pasta, exist_ok=True)

# Dicionário com os hiperparâmetros otimizados para o RT-DETR
hipers_rtdetr = {
    'optimizer': 'AdamW',
    'lr0': 0.0001,
    'lrf': 0.01,
    'momentum': 0.9,
    'weight_decay': 0.0001,
    'warmup_epochs': 3.0,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'degrees': 0.0,
    'translate': 0.1,
    'scale': 0.5,
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.0,
    'fliplr': 0.5,
    'mosaic': 0.5,
    'mixup': 0.0
}

# Salvando o arquivo no formato YAML
with open(caminho_arquivo, 'w') as f:
    yaml.dump(hipers_rtdetr, f, default_flow_style=False, sort_keys=False)

print(f"Arquivo salvo com sucesso em: {caminho_arquivo}")

Arquivo salvo com sucesso em: /content/drive/MyDrive/Lesc-Bee videos/modelo_v2/hyp_rtdetr.yaml


In [ ]:
!rm -rf /content/Abelhas-V3-1/**/*.cache

In [ ]:
from ultralytics import RTDETR

Fine_Tuning = False # Pega o melhor resultado e treina mais um pouco.
Resume = False      # Se False ele inicia um novo treinamento ao invés de continuar de onde parou o último treino.

if Fine_Tuning:
    # Carrega os pesos do melhor modelo treinado com RT-DETR
    model = RTDETR(f'{projeto_path}/{treino_name}/weights/best.pt')

    results = model.train(
        data=f'{dataset_paht}/data.yaml',
        cfg="/content/drive/MyDrive/Lesc-Bee videos/modelo_v2/hyp_rtdetr.yaml",
        epochs=23,
        batch=8,
        workers=6,
        cache='disk',
        imgsz=640,
        project=projeto_path,
        name=treino_name
    )

else:
    if Resume:
        # Carrega o "last.pt" do RT-DETR (último estado salvo antes de cair)
        model = RTDETR(f'{projeto_path}/{treino_name}/weights/last.pt')
        # Recupera exatamente onde parou
        results = model.train(resume=True)

    else:
        # Load a model
        model = RTDETR('rtdetr-l.pt')  # Carrega o modelo pré-treinado RT-DETR Large (também há o rtdetr-x.pt para Extra Large)

        results = model.train(
            data=f'{dataset_paht}/data.yaml',
            cfg="/content/drive/MyDrive/Lesc-Bee videos/modelo_v2/hyp_rtdetr.yaml",
            epochs=23,
            batch=12,
            workers=6,
            cache='disk',
            imgsz=640,
            project=projeto_path,
            name=treino_name
        )

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=disk, cfg=/content/drive/MyDrive/Lesc-Bee videos/modelo_v2/hyp_rtdetr.yaml, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Abelhas-V3-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=23, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.9, mosaic=0.5, multi_scale=0.0, name=rt_dert_28_02_20262, nbs=64, nms=False, opset=None, 

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/23      9.36G     0.8726      0.993     0.1862          6        640: 100% ━━━━━━━━━━━━ 630/630 1.0it/s 10:14
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 1.5it/s 21.5s
                   all        754       1013      0.612      0.217      0.124      0.051

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/23      9.48G     0.7732     0.7305     0.1261          9        640: 100% ━━━━━━━━━━━━ 630/630 1.1it/s 9:51
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 2.0it/s 15.7s
                   all        754       1013      0.753      0.388      0.347      0.138

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/23      9.44G     0.6904     0.5999     0.1113         23        640: 0% ──────────── 0/630  0.9s

...

      21/23      9.47G     0.3913     0.3885    0.06582         14        640: 100% ━━━━━━━━━━━━ 630/630 1.1it/s 9:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 2.1it/s 15.1s
                   all        754       1013      0.839      0.876      0.879      0.469

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      22/23      9.46G     0.3849     0.3899    0.06473         10        640: 100% ━━━━━━━━━━━━ 630/630 1.1it/s 9:31
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 2.1it/s 15.1s
                   all        754       1013       0.83      0.867      0.853       0.46

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      23/23      9.41G     0.3811     0.3871    0.06369         10        640: 100% ━━━━━━━━━━━━ 630/630 1.1it/s 9:35
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 2.1it/s 15.6s
                   all        754       1013      0.845      0.897      0.871      0.475

23 epochs completed in 3.847 hours.
Optimizer stripped from /content/drive/MyDrive/Lesc-Bee videos/modelo_rt_dert/rt_dert_28_02_20262/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/Lesc-Bee videos/modelo_rt_dert/rt_dert_28_02_20262/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/Lesc-Bee videos/modelo_rt_dert/rt_dert_28_02_20262/weights/best.pt...
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,987,850 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 1.

Realiza o teste do modelo com as imagens de teste

In [ ]:
# Atualizando o local do modelo apos o treino.
dataset_paht = "/content/Abelhas-V3-1"
projeto_path = "/content/drive/MyDrive/Lesc-Bee videos/modelo_rt_dert"
treino_name = "rt_dert_28_02_20262"

In [ ]:
from ultralytics import RTDETR
import os

# --- 1. Carregar o modelo treinado ---
# Note o "2" adicionado ao final do treino_name, conforme o diretório onde você relatou que foi salvo
MODEL_PATH = f'{projeto_path}/{treino_name}/weights/best.pt'

# Verifique se o arquivo do modelo existe antes de carregar
if not os.path.exists(MODEL_PATH):
    print(f"Erro: O arquivo do modelo não foi encontrado em: {MODEL_PATH}")
    print("Verifique se o treinamento foi concluído com sucesso e se o caminho está correto.")
else:
    # Carregue o seu melhor modelo usando a classe RTDETR
    model = RTDETR(MODEL_PATH)
    print("Modelo carregado com sucesso!")

    # --- 2. Executar a validação no conjunto de TESTE ---
    print("Iniciando a validação no conjunto de teste...")
    metrics = model.val(
        data=f'{dataset_paht}/data.yaml',
        split='test',
        imgsz=640,
        plots=True,
        project=projeto_path,
        name=f'{treino_name}2_inference'
    )
    print("Validação concluída!")

    # --- 3. Acessar e imprimir as métricas ---
    print("\n--- Métricas Gerais ---")
    print(f"mAP50-95 (geral): {metrics.box.map:.4f}")
    print(f"mAP50 (geral):    {metrics.box.map50:.4f}")
    print(f"mAP75 (geral):    {metrics.box.map75:.4f}")

    print("\n--- Métricas de Desempenho no Conjunto de Teste (por classe) ---")

    names = metrics.names
    map50_95 = metrics.box.maps
    map50 = metrics.box.all_ap[:, 0]
    map75 = metrics.box.all_ap[:, 5]

    precision = metrics.box.p
    recall = metrics.box.r

    for class_id, class_name in names.items():
        idx = int(class_id)

        print(f"\nClasse: {class_name} (id={idx})")
        print(f"  mAP50-95: {map50_95[idx]:.4f}")
        print(f"  mAP50:    {map50[idx]:.4f}")
        print(f"  mAP75:    {map75[idx]:.4f}")
        print(f"  Precision:{precision[idx]:.4f}")
        print(f"  Recall:   {recall[idx]:.4f}")

Modelo carregado com sucesso!
Iniciando a validação no conjunto de teste...
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
rt-detr-l summary: 310 layers, 31,987,850 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1351.8±390.9 MB/s, size: 48.6 KB)
val: Scanning /content/Abelhas-V3-1/test/labels... 504 images, 16 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 504/504 1.8Kit/s 0.3s
val: New cache created: /content/Abelhas-V3-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 51.0s/it 27:11
                   all        504        646      0.902      0.812      0.819      0.458
                   bee        488        609      0.956      0.975       0.98      0.655
                pollen         33         37      0.848      0.649      0.659      0.261
Speed: 8.0ms preprocess, 3219.9ms inference, 0.0ms loss, 0.3ms postprocess per image
R

In [ ]:
from ultralytics import RTDETR
import os
import glob
import random

# --- 1. Definir os caminhos ---
MODEL_PATH = f'{projeto_path}/{treino_name}/weights/best.pt'
TEST_IMAGES_DIR = f'{dataset_paht}/test/images'
NUM_IMAGES_TO_SELECT = 100 # Defina o número de imagens desejado

# --- 2. Encontrar todas as imagens e selecionar uma amostra aleatória ---
# Use um padrão para encontrar todos os tipos de imagem comuns
padrao_imagens = os.path.join(TEST_IMAGES_DIR, '*.jpg')
lista_completa_imagens = glob.glob(padrao_imagens)

# Verifique se alguma imagem foi encontrada
if not lista_completa_imagens:
    print(f"Nenhuma imagem encontrada na pasta: {TEST_IMAGES_DIR}")
else:
    print(f"Total de imagens encontradas: {len(lista_completa_imagens)}")

    # Garanta que não tentemos selecionar mais imagens do que as disponíveis
    if len(lista_completa_imagens) < NUM_IMAGES_TO_SELECT:
        print(f"Aviso: O número de imagens encontradas ({len(lista_completa_imagens)}) é menor que o solicitado ({NUM_IMAGES_TO_SELECT}).")
        print("Usando todas as imagens disponíveis.")
        amostra_de_imagens = lista_completa_imagens
    else:
        # Selecione imagens aleatoriamente da lista completa
        amostra_de_imagens = random.sample(lista_completa_imagens, NUM_IMAGES_TO_SELECT)

    print(f"Processando uma amostra aleatória de {len(amostra_de_imagens)} imagens...")

    # --- 3. Carregar o modelo ---
    model = RTDETR(MODEL_PATH)  # <-- Instanciando o modelo correto

    # --- 4. Executar a predição na amostra de imagens ---
    model.predict(
        source=amostra_de_imagens,
        save=True,
        imgsz=640,
        conf=0.1,  # Limiar de confiança de 10%
        project=projeto_path,
        name=f'{treino_name}_img_test'
    )

    print("\nPrevisão da amostra concluída!")
    print(f"As {len(amostra_de_imagens)} imagens com as detecções foram salvas na pasta: {projeto_path}/{treino_name}_img_test")

Total de imagens encontradas: 504
Processando uma amostra aleatória de 100 imagens...



In [ ]:
from ultralytics import RTDETR
import os
import shutil

# O comando com '!' funciona se você estiver rodando em um Jupyter Notebook / Google Colab
!rm -r /content/exportados_rpi5 # apaga o diretorio caso ja exista.

# --- Configurações ---
original_model_path = f'{projeto_path}/{treino_name}/weights/best.pt'

# 1. PREPARAÇÃO LOCAL
local_export_dir = '/content/exportados_rpi5'
os.makedirs(local_export_dir, exist_ok=True)
local_model_path = f'{local_export_dir}/best.pt'

print("--- Movendo modelo do Drive para o armazenamento local ---")
shutil.copy(original_model_path, local_model_path)
print("Modelo copiado para a máquina local com sucesso!")

imgsz = [640, 640]
data_yaml_path = f'{dataset_paht}/data.yaml'

print(f"\nCarregando modelo: {local_model_path}")
model = RTDETR(local_model_path)

# Função auxiliar para evitar que os arquivos se sobrescrevam
def renomear_exportacao(caminho_antigo, novo_nome):
    novo_caminho = os.path.join(local_export_dir, novo_nome)
    # Se já existir de uma execução anterior, deleta para evitar erro
    if os.path.exists(novo_caminho):
        if os.path.isdir(novo_caminho):
            shutil.rmtree(novo_caminho)
        else:
            os.remove(novo_caminho)
    os.rename(caminho_antigo, novo_caminho)
    print(f"-> Salvo com o nome correto: {novo_nome}")

# ============================================
# 1. Exportar para ONNX (FP32)
# ============================================
print("\n--- Exportando para ONNX (FP32) ---")
# CORREÇÃO AQUI: Mudamos o opset para 16
onnx_path = model.export(format='onnx', imgsz=imgsz, simplify=True, opset=16)
renomear_exportacao(onnx_path, 'modelo_onnx_fp32.onnx')

# ============================================
# 2. Exportar para ONNX (FP16)
# ============================================
print("\n--- Exportando para ONNX (FP16) ---")
# CORREÇÃO AQUI: Mudamos o opset para 16
onnx_fp16_path = model.export(format='onnx', imgsz=imgsz, simplify=True, half=True, opset=16)
renomear_exportacao(onnx_fp16_path, 'modelo_onnx_fp16.onnx')

# ============================================
# 3. Exportar para OpenVINO (INT8)
# ============================================
print("\n--- Exportando para OpenVINO (INT8) ---")
openvino_int8_path = model.export(format='openvino', imgsz=imgsz, int8=True, data=data_yaml_path)
renomear_exportacao(openvino_int8_path, 'int8_openvino_model')

# ============================================
# 4. Exportar para OpenVINO (FP16)
# ============================================
print("\n--- Exportando para OpenVINO (FP16) ---")
openvino_path = model.export(format='openvino', imgsz=imgsz, half=True)
renomear_exportacao(openvino_path, 'fp16_openvino_model')

# ============================================
# 5. Exportar para OpenVINO (FP32)
# ============================================
print("\n--- Exportando para OpenVINO (FP32) ---")
openvino_path = model.export(format='openvino', imgsz=imgsz)
renomear_exportacao(openvino_path, 'fp32_openvino_model')

# ============================================
# 6. Exportar para NCNN (FP32)
# ============================================
print("\n--- Exportando para NCNN (FP32) ---")
ncnn_fp32_path = model.export(format='ncnn', imgsz=imgsz)
renomear_exportacao(ncnn_fp32_path, 'ncnn_fp32')

# ============================================
# 7. Exportar para NCNN (FP16)
# ============================================
print("\n--- Exportando para NCNN (FP16) ---")
ncnn_fp16_path = model.export(format='ncnn', imgsz=imgsz, half=True)
renomear_exportacao(ncnn_fp16_path, 'ncnn_fp16')

# ============================================
# 8. Compactar e Mover para o Drive
# ============================================
print("\n--- Compactando os modelos exportados ---")
zip_base_name = '/content/modelos_exportados_rpi5'

# Compacta tudo que está dentro do 'local_export_dir'
shutil.make_archive(zip_base_name, 'zip', local_export_dir)
print(f"Arquivo ZIP criado localmente em: {zip_base_name}.zip")

print("\n--- Movendo arquivo ZIP para o Google Drive ---")
drive_dest_dir = f'{projeto_path}/{treino_name}'
os.makedirs(drive_dest_dir, exist_ok=True)

drive_zip_path = f'{drive_dest_dir}/exportados_rpi5.zip'
shutil.copy(f'{zip_base_name}.zip', drive_zip_path)

print(f"\n✅ SUCESSO! Modelos gerados com nomes únicos e salvos no Drive em: {drive_zip_path}")

--- Movendo modelo do Drive para o armazenamento local ---
Modelo copiado para a máquina local com sucesso!

Carregando modelo: /content/exportados_rpi5/best.pt

--- Exportando para ONNX (FP32) ---
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
rt-detr-l summary: 310 layers, 31,987,850 parameters, 0 gradients, 103.4 GFLOPs

PyTorch: starting from '/content/exportados_rpi5/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (63.1 MB)

ONNX: starting export with onnx 1.20.1 opset 16...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 16 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:714: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_graph_shape_type_inference(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1186: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. 

ONNX: slimming with onnxslim 0.1.86...
ONNX: export success ✅ 15.4s, saved as '/content/exportados_rpi5/best.onnx' (125.5 MB)

Export complete (27.8s)
Results saved to /content/exportados_rpi5
Predict:         yolo predict task=detect model=/content/exportados_rpi5/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/exportados_rpi5/best.onnx imgsz=640 data=/content/Abelhas-V3-1/data.yaml  
Visualize:       https://netron.app
-> Salvo com o nome correto: modelo_onnx_fp32.onnx

--- Exportando para ONNX (FP16) ---
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
rt-detr-l summary: 310 layers, 31,987,850 parameters, 0 gradients, 103.4 GFLOPs

PyTorch: starting from '/content/exportados_rpi5/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (63.1 MB)

ONNX: starting export with onnx 1.20.1 opset 16...
ONNX: slimming with onnxslim 0.1.86...
ONNX: converting to FP16...
ONNX: export success ✅ 17.3s, saved as

Output()

Output()

OpenVINO: export success ✅ 997.3s, saved as '/content/exportados_rpi5/best_int8_openvino_model/' (33.2 MB)

Export complete (1006.6s)
Results saved to /content/exportados_rpi5
Predict:         yolo predict task=detect model=/content/exportados_rpi5/best_int8_openvino_model imgsz=640 int8
Validate:        yolo val task=detect model=/content/exportados_rpi5/best_int8_openvino_model imgsz=640 data=/content/Abelhas-V3-1/data.yaml int8 
Visualize:       https://netron.app
-> Salvo com o nome correto: int8_openvino_model

--- Exportando para OpenVINO (FP16) ---
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
rt-detr-l summary: 310 layers, 31,987,850 parameters, 0 gradients, 103.4 GFLOPs

PyTorch: starting from '/content/exportados_rpi5/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (63.1 MB)

OpenVINO: starting export with openvino 2026.0.0-20965-c6d6a13a886-releases/2026/0...
OpenVINO: export success ✅ 27.5s, saved as '/co

Teste dos modelos exportados com as imagens de teste.

In [ ]:
import os
import shutil
import pandas as pd
from ultralytics import RTDETR  # <-- Atualizado para RTDETR

dataset_paht = "/content/Abelhas-V3-1"
local_export_dir = '/content/exportados_rpi5'

projeto_path = "/content/drive/MyDrive/Lesc-Bee videos/modelo_rt_dert"
treino_name = "rt_dert_28_02_20262"  # <-- Corrigido o nome da variável sem aspas no meio

# ==========================================
# 1. Definir os modelos com os nomes corretos
# ==========================================
modelos_para_testar = {
    "PyTorch_Original": f"{local_export_dir}/best.pt",
    "ONNX_FP32": f"{local_export_dir}/modelo_onnx_fp32.onnx",
    "ONNX_FP16": f"{local_export_dir}/modelo_onnx_fp16.onnx",
    "OpenVINO_FP32": f"{local_export_dir}/fp32_openvino_model",
    "OpenVINO_FP16": f"{local_export_dir}/fp16_openvino_model",
    "OpenVINO_INT8": f"{local_export_dir}/int8_openvino_model"
}

# ==========================================
# 2. Rodar os testes e extrair métricas
# ==========================================
resultados = []

for nome_modelo, caminho_modelo in modelos_para_testar.items():
    if not os.path.exists(caminho_modelo):
        print(f"⚠️ Pulando {nome_modelo}: Arquivo/Pasta não encontrado em {caminho_modelo}")
        continue

    print(f"\n========================================")
    print(f"🚀 Iniciando teste: {nome_modelo}")
    print(f"========================================")

    try:
        # Instancia o modelo usando a classe RTDETR
        model = RTDETR(caminho_modelo)

        tempo_gpu = None
        tempo_cpu = None
        metrics_base = None

        # --- Teste na GPU ---
        print("   -> Testando inferência na GPU...")
        try:
            metrics_gpu = model.val(
                data=f'{dataset_paht}/data.yaml',
                split='test',
                imgsz=640,
                device=0,  # Força o uso da GPU
                plots=False,
                verbose=False
            )
            tempo_gpu = round(metrics_gpu.speed['inference'], 2)
            metrics_base = metrics_gpu # Guarda as métricas principais
            print(f"      Tempo GPU: {tempo_gpu} ms/imagem")
        except Exception as e:
            print(f"      ⚠️ GPU não suportada ou falhou para formato {nome_modelo}. Erro: {e}")

        # --- Teste na CPU ---
        print("   -> Testando inferência na CPU...")
        try:
            metrics_cpu = model.val(
                data=f'{dataset_paht}/data.yaml',
                split='test',
                imgsz=640,
                device='cpu',  # Força o uso da CPU
                plots=False,
                verbose=False
            )
            tempo_cpu = round(metrics_cpu.speed['inference'], 2)
            if metrics_base is None:
                metrics_base = metrics_cpu # Se a GPU falhou, usa a precisão da CPU
            print(f"      Tempo CPU: {tempo_cpu} ms/imagem")
        except Exception as e:
            print(f"      ⚠️ CPU falhou para formato {nome_modelo}. Erro: {e}")


        # --- Extrair Métricas de Acurácia ---
        # Só prossegue se conseguiu rodar pelo menos na CPU ou GPU
        if metrics_base is not None:
            names = metrics_base.names
            map50_95 = metrics_base.box.maps
            precision = metrics_base.box.p
            recall = metrics_base.box.r
            map50_classes = metrics_base.box.all_ap[:, 0]
            map75_classes = metrics_base.box.all_ap[:, 5]

            for class_id, class_name in names.items():
                idx = int(class_id)
                resultados.append({
                    "Modelo": nome_modelo,
                    "Classe": class_name,
                    "Precision": round(precision[idx], 4),
                    "Recall": round(recall[idx], 4),
                    "mAP50": round(map50_classes[idx], 4),
                    "mAP75": round(map75_classes[idx], 4),
                    "mAP50-95": round(map50_95[idx], 4),
                    "Inferência_GPU (ms)": tempo_gpu if tempo_gpu else "N/A",
                    "Inferência_CPU (ms)": tempo_cpu if tempo_cpu else "N/A"
                })

            print(f"✅ {nome_modelo} testado com sucesso!")
        else:
            print(f"❌ Falha total ao testar {nome_modelo}. Nenhuma métrica extraída.")

    except Exception as e:
        print(f"❌ Erro crítico ao testar {nome_modelo}: {e}")

# ==========================================
# 3. Gerar e Salvar o CSV
# ==========================================
print("\n--- Compilando resultados no CSV ---")
df_resultados = pd.DataFrame(resultados)
print(df_resultados.head(12))

caminho_csv = '/content/comparativo_modelos.csv'
df_resultados.to_csv(caminho_csv, index=False)

# Garantir que a pasta de destino exista no Drive antes de copiar
drive_dest_dir = f'{projeto_path}/{treino_name}'
os.makedirs(drive_dest_dir, exist_ok=True)

# Copia o arquivo CSV finalizado para o Google Drive
drive_csv_path = f'{drive_dest_dir}/comparativo_modelos.csv'
shutil.copy(caminho_csv, drive_csv_path)

print(f"\n✅ Tudo pronto! CSV salvo no seu Drive em: {drive_csv_path}")


🚀 Iniciando teste: PyTorch_Original
   -> Testando inferência na GPU...
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu 
      ⚠️ GPU não suportada ou falhou para formato PyTorch_Original. Erro: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: 
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.

   -> Testando inferência na CPU...
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
rt-detr-l summary: 310 layers, 31,987,850 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 13.3±7.7 MB/s, size: 44.0 KB)
val: Scanning /content/Abelhas-V3-1/test/labels.cache... 504 images, 16 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 504/504 7.6Mit/s 0.0s
           